In [36]:
import glob, os, contextlib
from pathlib import Path
import numpy as np
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()
W, H = 400, 300

OUT = Path("g7_debt_outputs"); (OUT / "data").mkdir(parents=True, exist_ok=True)

@contextlib.contextmanager
def in_dir(path):
    old = Path.cwd(); os.chdir(path)
    try: yield
    finally: os.chdir(old)

def save(chart, name):
    """Try the likely ecostyles call patterns; fall back to Altair's own export."""
    attempts = [
        lambda: styles.save(chart, str(OUT), name=name, svg=True),
        lambda: styles.save(chart, name=name, svg=True, path=str(OUT)),
        lambda: styles.save(chart, str(OUT / name), svg=True),
    ]
    last = None
    for i, fn in enumerate(attempts, 1):
        try:
            fn(); print(f"saved via ecostyles, attempt {i}: {name}"); return
        except TypeError as err:
            last = err
    print(f"ecostyles attempts failed ({last}); using Altair export")
    stem = OUT / name
    chart.save(f"{stem}.json"); chart.save(f"{stem}.svg"); chart.save(f"{stem}.png", scale_factor=2)

def dump(df, name):
    df.to_csv(OUT / "data" / f"{name}.csv")

# Data log: IMF.RES:WEO_2026_APR_VINTAGE, downloaded 20 Oct 2026. Projections = 2026 onward.

In [37]:
RAW = glob.glob("../Data/dataset_*WEO_2026_APR_VINTAGE*.csv")[0]
print(RAW)
CODES = ["GGXWDG_NGDP", "GGXWDN_NGDP", "GGXONLB_NGDP", "GGXCNL_NGDP",
         "GGR_NGDP", "GGX_NGDP", "NGDP_D", "NGDP_RPCH", "NGDPD"]

df = pd.read_csv(RAW, encoding="utf-8-sig")
df["code"] = df.SERIES_CODE.str.split(".").str[1]
years = [c for c in df.columns if c.isdigit()]
long = (df[df.code.isin(CODES)]
        .melt(id_vars=["COUNTRY", "code"], value_vars=years, var_name="year", value_name="v"))
long["year"] = long.year.astype(int)
long["v"] = pd.to_numeric(long.v, errors="coerce")
p = (long.pivot_table(index=["COUNTRY", "year"], columns="code", values="v")
         .reset_index().rename(columns={"COUNTRY": "country"})
         .sort_values(["country", "year"]).reset_index(drop=True))

p["pb"], p["ob"] = p.GGXONLB_NGDP, p.GGXCNL_NGDP
p["interest"] = p.pb - p.ob                                   # net interest, % GDP
p["g"] = (1 + p.NGDP_RPCH/100) * (1 + p.groupby("country")["NGDP_D"].pct_change()) - 1
p["debt_lag"] = p.groupby("country")["GGXWDG_NGDP"].shift(1)
p["pb_star"] = p.interest - p.g/(1 + p.g) * p.debt_lag        # debt-stabilising primary balance
p["gap"] = p.pb - p.pb_star
p["prim_spend"] = p.GGR_NGDP - p.pb

../Data/dataset_2026-10-20T09_42_49.749383602Z_DEFAULT_INTEGRATION_IMF.RES_WEO_2026_APR_VINTAGE_1.0.0.csv


In [38]:
pre, post = (2014, 2019), (2024, 2029)
wm = lambda col, a, b: p[p.year.between(a, b)].groupby("country")[col].mean()
dec = pd.DataFrame({
    "revenue":       wm("GGR_NGDP", *post) - wm("GGR_NGDP", *pre),
    "primary_spend": -(wm("prim_spend", *post) - wm("prim_spend", *pre)),
    "snowball":      -(wm("pb_star", *post) - wm("pb_star", *pre)),
})
dec["gap_change"] = dec.sum(axis=1)
print(dec.round(2))

w = p.pivot(index="year", columns="country", values="NGDPD")
share = w.div(w.sum(axis=1), axis=0)
gap = p.pivot(index="year", columns="country", values="gap")
weighted = (gap * share).sum(axis=1)

                revenue  primary_spend  snowball  gap_change
country                                                     
Canada             1.74          -3.96      1.80       -0.42
France            -1.30           0.49      0.25       -0.57
Germany            1.74          -6.36     -0.50       -5.13
Italy              0.73          -1.14      1.28        0.87
Japan              2.60          -1.60      3.04        4.04
United Kingdom     4.02          -3.45     -0.37        0.20
United States     -0.28          -0.75     -0.53       -1.56


In [39]:
d = dec.rename(columns={"revenue": "Revenue", "primary_spend": "Primary spending",
                        "snowball": "Interest-growth", "gap_change": "Total"})
comps = ["Primary spending", "Interest-growth", "Revenue"]
order = d.Total.sort_values(ascending=False).index.tolist()
longd = d.reset_index().melt(id_vars=["country", "Total"], value_vars=comps,
                             var_name="component", value_name="pp")
bars = alt.Chart(longd).mark_bar(height=16).encode(
    y=alt.Y("country:N", sort=order, title=None),
    x=alt.X("pp:Q", title="Percentage points of GDP",
            scale=alt.Scale(domain=[-7.5, 9]), axis=alt.Axis(values=list(range(-8, 7, 2)))),
    color=alt.Color("component:N", title=None, sort=comps,
                    legend=alt.Legend(orient="top", direction="horizontal")))
tot = d.reset_index()[["country", "Total"]]
dots = alt.Chart(tot).mark_point(filled=True, color="black", size=60).encode(
    y=alt.Y("country:N", sort=order), x="Total:Q")
lab = alt.Chart(tot).mark_text(align="left", fontSize=12, fontWeight="bold").encode(
    y=alt.Y("country:N", sort=order), x=alt.datum(6.8), text=alt.Text("Total:Q", format="+.1f"))
fig1 = (bars + dots + lab).properties(width=W, height=H,
        title="Change in G7 fiscal gaps, 2014–19 to 2024–29")
save(fig1, "g7_fig01_fiscal_gap_decomposition"); dump(dec, "g7_fig01_fiscal_gap_decomposition")
fig1

saved via ecostyles, attempt 1: g7_fig01_fiscal_gap_decomposition


alt.LayerChart(...)

In [40]:
rows = []
for name, (a, b) in {"2014–19": pre, "2024–29": post}.items():
    rows.append({"period": name, "measure": "GDP-weighted", "gap": weighted.loc[a:b].mean()})
    rows.append({"period": name, "measure": "Median country", "gap": gap.loc[a:b].mean().median()})
wm_df = pd.DataFrame(rows)
base = alt.Chart(wm_df).encode(
    x=alt.X("period:N", title=None, axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset("measure:N"),
    y=alt.Y("gap:Q", title="% of GDP", scale=alt.Scale(domain=[-1.2, 0.4])),
    color=alt.Color("measure:N", title=None, legend=alt.Legend(orient="top", direction="horizontal")))
txt = alt.Text("gap:Q", format="+.2f")
pos = base.mark_text(dy=-8, fontSize=11).encode(text=txt).transform_filter("datum.gap > 0")
neg = base.mark_text(dy=13, fontSize=11).encode(text=txt).transform_filter("datum.gap <= 0")
fig2 = (base.mark_bar() + pos + neg).properties(width=W, height=H,
        title="G7 fiscal gap: weighted average vs median country")
save(fig2, "g7_fig02_weighted_vs_median"); dump(wm_df.set_index("period"), "g7_fig02_weighted_vs_median")
fig2

saved via ecostyles, attempt 1: g7_fig02_weighted_vs_median


alt.LayerChart(...)

In [41]:
dA = p[p.year.between(2000, 2031)][["country", "year", "GGXWDG_NGDP"]].rename(columns={"GGXWDG_NGDP": "debt"})
xsA = alt.Scale(domain=[2000, 2039], zero=False)
axA = alt.Axis(format="d", values=[2000, 2010, 2020, 2030], grid=False)
proj = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
    x=alt.X("a:Q", scale=xsA, title=None, axis=axA), x2="b:Q")
ln = alt.Chart(dA).mark_line(strokeWidth=1.8).encode(
    x=alt.X("year:Q", title=None, scale=xsA, axis=axA), y=alt.Y("debt:Q", title="% of GDP"),
    color=alt.Color("country:N", legend=None))
last = dA[dA.year == 2031].sort_values("debt").copy()
def spread(vals, gap_=11):                      # stops end-of-line labels colliding
    v = np.array(vals, float); out = v.copy()
    for i in range(1, len(out)): out[i] = max(out[i], out[i-1] + gap_)
    return out - (out.mean() - v.mean())
last["ylab"] = spread(last.debt.values)
lab3 = alt.Chart(last).mark_text(align="left", dx=4, fontSize=11).encode(
    x=alt.X("year:Q", scale=xsA), y="ylab:Q", text="country:N", color=alt.Color("country:N", legend=None))
fig3 = (proj + ln + lab3).properties(width=W, height=H, title="General government gross debt, G7, 2000–2031")
save(fig3, "g7_fig03_debt_paths")
dump(dA.pivot(index="year", columns="country", values="debt"), "g7_fig03_debt_paths")
fig3

saved via ecostyles, attempt 1: g7_fig03_debt_paths


alt.LayerChart(...)

In [42]:
def gapchart(country, label=None, start=2005):
    label = label or country
    s = p[(p.country == country) & p.year.between(start, 2031)][["year", "pb", "pb_star"]]
    s = s.rename(columns={"pb": "Primary balance", "pb_star": "Debt-stabilising primary balance"}
                ).melt("year", var_name="series", value_name="v")
    xs = alt.Scale(domain=[start, 2031], zero=False)
    ax = alt.Axis(format="d", values=[2005, 2010, 2015, 2020, 2025, 2030], grid=False)
    pr = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
        x=alt.X("a:Q", scale=xs, title=None, axis=ax), x2="b:Q")
    zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[1, 5], color="#94a3b8").encode(y="y:Q")
    line = alt.Chart(s).mark_line(strokeWidth=1.8).encode(
        x=alt.X("year:Q", title=None, scale=xs, axis=ax), y=alt.Y("v:Q", title="% of GDP"),
        color=alt.Color("series:N", title=None, legend=alt.Legend(orient="top", direction="vertical", labelLimit=300)))
    return (pr + zero + line).properties(width=W, height=H,
            title=f"{label}: primary balance vs debt-stabilising balance")

fig4 = gapchart("United Kingdom", label="UK")
save(fig4, "g7_fig04_uk_pb_vs_stabilising")
dump(p[p.country == "United Kingdom"].set_index("year")[["pb", "pb_star"]], "g7_fig04_uk_pb_vs_stabilising")
fig4

saved via ecostyles, attempt 1: g7_fig04_uk_pb_vs_stabilising


alt.LayerChart(...)

In [43]:
p["nd_lag"] = p.groupby("country")["GGXWDN_NGDP"].shift(1)
p["c_primary"]  = -p.pb
p["c_interest"] = p.interest
p["c_growth"]   = -p.g/(1 + p.g) * p.nd_lag
p["c_sfa"] = (p.GGXWDN_NGDP - p.nd_lag) - (p.c_primary + p.c_interest + p.c_growth)
hist = p[p.year.between(2020, 2025)].groupby("country")[["c_primary", "c_interest", "c_growth", "c_sfa"]].sum()
hist["Total"] = hist.sum(axis=1)
print(hist.round(1))

h = hist.rename(columns={"c_primary": "Primary deficit", "c_interest": "Net interest",
                         "c_growth": "Growth & inflation", "c_sfa": "Residual"})
comps5 = ["Primary deficit", "Net interest", "Growth & inflation", "Residual"]
order5 = h.Total.sort_values().index.tolist()
lg = h.reset_index().melt(id_vars=["country", "Total"], value_vars=comps5, var_name="driver", value_name="pp")
bars5 = alt.Chart(lg).mark_bar(height=16).encode(
    y=alt.Y("country:N", sort=order5, title=None),
    x=alt.X("pp:Q", title="Percentage points of GDP, 2020–25 cumulative",
            scale=alt.Scale(domain=[-42, 75]), axis=alt.Axis(values=list(range(-40, 61, 20)))),
    color=alt.Color("driver:N", title=None, sort=comps5,
                    legend=alt.Legend(orient="top", direction="horizontal", columns=2)))
t5 = h.reset_index()[["country", "Total"]]
dots5 = alt.Chart(t5).mark_point(filled=True, color="black", size=60).encode(
    y=alt.Y("country:N", sort=order5), x="Total:Q")
lbl5 = alt.Chart(t5).mark_text(align="left", fontSize=12, fontWeight="bold").encode(
    y=alt.Y("country:N", sort=order5), x=alt.datum(63), text=alt.Text("Total:Q", format="+.1f"))
fig5 = (bars5 + dots5 + lbl5).properties(width=W, height=H, title="What drove the rise in G7 net debt, 2019–25")
save(fig5, "g7_fig05_net_debt_drivers"); dump(hist, "g7_fig05_net_debt_drivers")
fig5

code            c_primary  c_interest  c_growth  c_sfa  Total
country                                                      
Canada               18.2        -0.8      -4.8  -11.2    1.5
France               26.6        10.0     -20.5    3.7   19.8
Germany              13.5         3.8     -10.5    0.6    7.3
Italy                18.8        21.1     -28.9   -6.3    4.7
Japan                22.6         2.1     -22.4  -14.0  -11.7
United Kingdom       28.7        13.8     -26.6    2.8   18.7
United States        34.6        17.3     -32.5   -4.3   15.1
saved via ecostyles, attempt 1: g7_fig05_net_debt_drivers


alt.LayerChart(...)

In [44]:
pre_pb = p[p.year.between(2017, 2019)].groupby("country")["pb"].mean()
dd = p[p.year == 2025].set_index("country")["GGXWDG_NGDP"] - p[p.year == 2019].set_index("country")["GGXWDG_NGDP"]
sc = pd.DataFrame({"pre_pb": pre_pb, "debt_change": dd}).reset_index()
sc["group"] = np.where(sc.country == "United Kingdom", "United Kingdom", "Other G7")
pts = alt.Chart(sc).mark_point(filled=True, size=70).encode(
    x=alt.X("pre_pb:Q", title="Average primary balance, 2017–19 (% of GDP)",
            scale=alt.Scale(domain=[-4, 3.5]), axis=alt.Axis(grid=False)),
    y=alt.Y("debt_change:Q", title="Change in gross debt, 2019–25 (pp of GDP)", scale=alt.Scale(domain=[-2, 26])),
    color=alt.Color("group:N", legend=None, sort=["United Kingdom", "Other G7"]))
b6 = alt.Chart(sc).encode(x="pre_pb:Q", y="debt_change:Q", text="country:N")
txr = b6.mark_text(align="left", dx=7, fontSize=11).transform_filter("datum.country != 'France'")
txl = b6.mark_text(align="right", dx=-7, fontSize=11).transform_filter("datum.country == 'France'")
fig6 = (pts + txr + txl).properties(width=W, height=H, title="Debt rise since 2019 vs pre-shock primary balance")
save(fig6, "g7_fig06_buffer_scatter"); dump(sc.set_index("country"), "g7_fig06_buffer_scatter")
fig6

saved via ecostyles, attempt 1: g7_fig06_buffer_scatter


alt.LayerChart(...)

In [45]:
e = (p[p.year == 2031].set_index("country")["pb"] - p[p.year == 2025].set_index("country")["pb"]).rename("swing").reset_index()
e["group"] = np.where(e.country == "United Kingdom", "United Kingdom", "Other G7")
oE = e.sort_values("swing", ascending=False).country.tolist()
xs7 = alt.Scale(domain=[-2.6, 5])
bar7 = alt.Chart(e).mark_bar(height=16).encode(
    y=alt.Y("country:N", sort=oE, title=None),
    x=alt.X("swing:Q", title="Percentage points of GDP", scale=xs7),
    color=alt.Color("group:N", legend=None, sort=["United Kingdom", "Other G7"]))
lab7 = alt.Chart(e).encode(y=alt.Y("country:N", sort=oE), x=alt.X("swing:Q", scale=xs7),
                           text=alt.Text("swing:Q", format="+.1f"))
lpos = lab7.mark_text(align="left", dx=5, fontSize=12, fontWeight="bold").transform_filter("datum.swing >= 0")
lneg = lab7.mark_text(align="right", dx=-5, fontSize=12, fontWeight="bold").transform_filter("datum.swing < 0")
fig7 = (bar7 + lpos + lneg).properties(width=W, height=H, title="Projected change in primary balance, 2025 to 2031")
save(fig7, "g7_fig07_projected_pb_swing"); dump(e.set_index("country"), "g7_fig07_projected_pb_swing")
fig7

saved via ecostyles, attempt 1: g7_fig07_projected_pb_swing


alt.LayerChart(...)

In [46]:
summary = pd.DataFrame({
    "gross_debt_2019": p[p.year == 2019].set_index("country")["GGXWDG_NGDP"],
    "gross_debt_2025": p[p.year == 2025].set_index("country")["GGXWDG_NGDP"],
    "gross_debt_2031": p[p.year == 2031].set_index("country")["GGXWDG_NGDP"],
    "pb_2017_19_avg": pre_pb,
    "pb_2025": p[p.year == 2025].set_index("country")["pb"],
    "pb_2031": p[p.year == 2031].set_index("country")["pb"],
}).round(2)
dump(summary, "g7_summary_table")
dump(p, "g7_panel_full")
print(sorted(os.listdir(OUT)))

['data', 'g7_fig01_fiscal_gap_decomposition.json', 'g7_fig01_fiscal_gap_decomposition.png', 'g7_fig01_fiscal_gap_decomposition.svg', 'g7_fig02_weighted_vs_median.json', 'g7_fig02_weighted_vs_median.png', 'g7_fig02_weighted_vs_median.svg', 'g7_fig03_debt_paths.json', 'g7_fig03_debt_paths.png', 'g7_fig03_debt_paths.svg', 'g7_fig04_uk_pb_vs_stabilising.json', 'g7_fig04_uk_pb_vs_stabilising.png', 'g7_fig04_uk_pb_vs_stabilising.svg', 'g7_fig05_net_debt_drivers.json', 'g7_fig05_net_debt_drivers.png', 'g7_fig05_net_debt_drivers.svg', 'g7_fig06_buffer_scatter.json', 'g7_fig06_buffer_scatter.png', 'g7_fig06_buffer_scatter.svg', 'g7_fig07_projected_pb_swing.json', 'g7_fig07_projected_pb_swing.png', 'g7_fig07_projected_pb_swing.svg', 'g7_fig08_shock_drift.json', 'g7_fig08_shock_drift.png', 'g7_fig08_shock_drift.svg', 'g7_fig09_return_to_norm.json', 'g7_fig09_return_to_norm.png', 'g7_fig09_return_to_norm.svg', 'g7_fig10_uk_revenue_spending.json', 'g7_fig10_uk_revenue_spending.png', 'g7_fig10_uk_re

In [47]:

p["nd_lag"] = p.groupby("country")["GGXWDN_NGDP"].shift(1)
p["c_growth"] = -p.g / (1 + p.g) * p.nd_lag
p["c_sfa"] = (p.GGXWDN_NGDP - p.nd_lag) - (-p.pb + p.interest + p.c_growth)

def decompose(start=2007, end=2025, base=(2003, 2007),
              windows=((2008, 2010), (2020, 2022)), covid_base=None):
    out = []
    for c, s in p.groupby("country"):
        s = s.set_index("year")
        b0 = s.loc[base[0]:base[1], "pb"].mean()                      # pre-shock primary-balance norm
        b1 = s.loc[covid_base[0]:covid_base[1], "pb"].mean() if covid_base else b0
        baseline = shock = drift = 0.0
        for y in range(start + 1, end + 1):
            b = b1 if y >= windows[1][0] else b0
            baseline += -b                                            # what the pre-shock norm alone would add
            dev = -(s.loc[y, "pb"] - b)                               # deviation from the norm (+ adds to debt)
            if any(lo <= y <= hi for lo, hi in windows):
                shock += dev                                          # inside a shock window
            else:
                drift += dev                                          # outside the windows
        snow = s.loc[start + 1:end, ["interest", "c_growth"]].sum().sum()
        sfa = s.loc[start + 1:end, "c_sfa"].sum()
        total = s.loc[end, "GGXWDN_NGDP"] - s.loc[start, "GGXWDN_NGDP"]
        out.append(dict(country=c, baseline=baseline, shocks=shock, drift=drift,
                        snowball=snow, residual=sfa, total=total))
    return pd.DataFrame(out).set_index("country")

sd = decompose()
print((sd.baseline + sd.shocks + sd.drift + sd.snowball + sd.residual - sd.total).abs().max())  # should be ~0
print(sd.round(1))

alts = {
    "main":                          {},
    "windows 2008-11 & 2020-23":     dict(windows=((2008, 2011), (2020, 2023))),
    "windows 2009-10 & 2020-21":     dict(windows=((2009, 2010), (2020, 2021))),
    "baseline 2000-07":              dict(base=(2000, 2007)),
    "COVID-era baseline = 2017-19":  dict(covid_base=(2017, 2019)),
}
sens_shock = pd.DataFrame({k: decompose(**kw).shocks for k, kw in alts.items()}).round(1)
sens_drift = pd.DataFrame({k: decompose(**kw).drift for k, kw in alts.items()}).round(1)
print(sens_shock); print(sens_drift)
dump(sd, "g7_shock_drift_decomposition"); dump(sens_shock, "g7_shock_sensitivity"); dump(sens_drift, "g7_drift_sensitivity")

# ---------- FIG 8 ----------
h8 = sd.rename(columns={"baseline": "Pre-shock norm", "snowball": "Interest-growth",
                        "residual": "Residual", "shocks": "Shocks", "drift": "Drift", "total": "Total"})
comps8 = ["Pre-shock norm", "Interest-growth", "Residual", "Shocks", "Drift"]
order8 = h8.Total.sort_values(ascending=False).index.tolist()
lg8 = h8.reset_index().melt(id_vars=["country", "Total"], value_vars=comps8, var_name="driver", value_name="pp")
bars8 = alt.Chart(lg8).mark_bar(height=16).encode(
    y=alt.Y("country:N", sort=order8, title=None),
    x=alt.X("pp:Q", title="Percentage points of GDP, 2008–25 cumulative",
            scale=alt.Scale(domain=[-90, 125]), axis=alt.Axis(values=list(range(-80, 81, 40)))),
    color=alt.Color("driver:N", title=None, sort=comps8,
                    legend=alt.Legend(orient="top", direction="horizontal", columns=3)))
t8 = h8.reset_index()[["country", "Total"]]
dots8 = alt.Chart(t8).mark_point(filled=True, color="black", size=60).encode(
    y=alt.Y("country:N", sort=order8), x="Total:Q")
lbl8 = alt.Chart(t8).mark_text(align="left", fontSize=12, fontWeight="bold").encode(
    y=alt.Y("country:N", sort=order8), x=alt.datum(100), text=alt.Text("Total:Q", format="+.1f"))
fig8 = (bars8 + dots8 + lbl8).properties(width=W, height=H, title="Shocks, drift and the pre-shock norm in G7 net debt, 2007–25")
save(fig8, "g7_fig08_shock_drift"); dump(sd, "g7_fig08_shock_drift")
fig8

1.4210854715202004e-14
                baseline  shocks  drift  snowball  residual  total
country                                                           
Canada             -40.4    33.6   34.9      -8.1     -32.0  -11.9
France              16.1    21.2   18.2      -4.3      -1.5   49.7
Germany             -0.5     9.2  -11.4      -6.8       3.5   -5.9
Italy              -23.1    22.4    5.3      28.5      -1.8   31.3
Japan               66.2    15.8   -2.4     -18.0     -18.6   43.0
United Kingdom      25.5    28.5   18.1     -17.7       1.4   55.9
United States       25.8    38.5   26.5     -19.9     -18.3   52.6
                main  windows 2008-11 & 2020-23  windows 2009-10 & 2020-21  baseline 2000-07  COVID-era baseline = 2017-19
country                                                                                                                   
Canada          33.6                       40.8                       29.8              37.0                          27.5
Franc

alt.LayerChart(...)

In [48]:
s = p[p.country == "United Kingdom"].set_index("year")
b = s.loc[2003:2007, "pb"].mean()
t = pd.DataFrame({
    "primary_balance": s.pb,
    "deviation_from_norm": -(s.pb - b),
    "net_interest": s.interest,
    "growth_effect": s.c_growth,
    "residual": s.c_sfa,
    "change_in_net_debt": s.GGXWDN_NGDP - s.nd_lag,
}).loc[2008:2025]
t["bucket"] = ["shock" if (2008 <= y <= 2010 or 2020 <= y <= 2022) else "drift" for y in t.index]
print(round(b, 2)); print(t.round(2))

-1.42
      primary_balance  deviation_from_norm  net_interest  growth_effect  residual  change_in_net_debt bucket
year                                                                                                        
2008            -3.50                 2.09          1.54          -1.19      3.02                6.88  shock
2009            -8.51                 7.09          1.46           1.23      1.85               13.05  shock
2010            -6.72                 5.30          2.53          -2.02      3.40               10.63  shock
2011            -4.63                 3.21          2.80          -2.22     -0.78                4.43  drift
2012            -5.62                 4.20          2.34          -2.19     -2.97                2.79  drift
2013            -3.90                 2.49          1.39          -2.85     -1.38                1.07  drift
2014            -3.65                 2.23          1.82          -3.52     -0.32                1.63  drift
2015         

In [49]:
BASE = (2003, 2007)
norm = p[p.year.between(*BASE)].groupby("country")["pb"].mean()
p["pb_norm"] = p.country.map(norm)
p["dev"] = p.pb - p.pb_norm                       # primary balance minus pre-shock norm, pp of GDP

def first_return(s, after):
    ys = s[(s.index > after) & (s >= 0)].index
    return int(ys[0]) if len(ys) else np.nan

rows = []
for c, g in p.groupby("country"):
    s = g.set_index("year")["dev"]
    rows.append(dict(country=c, norm=norm[c],
                     worst_gfc=s.loc[2008:2012].min(), back_after_gfc=first_return(s, 2010),
                     worst_covid=s.loc[2020:2022].min(), back_after_covid=first_return(s, 2022),
                     dev_2025=s.loc[2025], dev_2031=s.loc[2031]))
ret = pd.DataFrame(rows).set_index("country")
print(ret.round(1))
dump(ret, "g7_return_to_norm")

# ---------- FIG 9 ----------
d9 = p[p.year.between(2005, 2031)][["country", "year", "dev"]].copy()
xs9 = alt.Scale(domain=[2005, 2038], zero=False)
ax9 = alt.Axis(format="d", values=[2005, 2010, 2015, 2020, 2025, 2030], grid=False)
proj9 = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
    x=alt.X("a:Q", scale=xs9, title=None, axis=ax9), x2="b:Q")
zero9 = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[1, 5], color="#94a3b8").encode(y="y:Q")
ln9 = alt.Chart(d9).mark_line(strokeWidth=1.8).encode(
    x=alt.X("year:Q", title=None, scale=xs9, axis=ax9),
    y=alt.Y("dev:Q", title="pp of GDP", scale=alt.Scale(domain=[-14, 6])),
    color=alt.Color("country:N", legend=None))

def spread2(vals, gap_=0.9):                       # keeps end-of-line labels from colliding
    v = np.array(vals, float); out = v.copy()
    for i in range(1, len(out)): out[i] = max(out[i], out[i - 1] + gap_)
    return out - (out.mean() - v.mean())
last9 = d9[d9.year == 2031].sort_values("dev").copy()
last9["ylab"] = spread2(last9.dev.values)
lab_base = alt.Chart(last9).encode(
    x=alt.X("year:Q", scale=xs9), y="ylab:Q", text="country:N", color=alt.Color("country:N", legend=None))
lab9 = lab_base.mark_text(align="left", dx=4, fontSize=11).transform_filter("datum.country != 'United Kingdom'")
lab9uk = lab_base.mark_text(align="left", dx=4, fontSize=11, fontWeight="bold").transform_filter("datum.country == 'United Kingdom'")

fig9 = (proj9 + zero9 + ln9 + lab9 + lab9uk).properties(
    width=W, height=H, title="Primary balance relative to pre-crisis norm, G7")
save(fig9, "g7_fig09_return_to_norm"); dump(d9.pivot(index="year", columns="country", values="dev"), "g7_fig09_return_to_norm")
fig9

                norm  worst_gfc  back_after_gfc  worst_covid  back_after_covid  dev_2025  dev_2031
country                                                                                           
Canada           2.2       -6.1             NaN        -12.7               NaN      -4.3      -3.1
France          -0.9       -4.2          2030.0         -6.8            2030.0      -2.2       1.3
Germany          0.0       -2.3          2011.0         -3.9               NaN      -1.8      -2.2
Italy            1.3       -2.2          2012.0         -7.4            2028.0      -0.8       0.3
Japan           -3.7       -4.9          2015.0         -4.7            2023.0       2.8       1.4
United Kingdom  -1.4       -7.1          2017.0        -10.4            2026.0      -1.3       2.8
United States   -1.4       -9.9          2022.0        -10.6               NaN      -1.7      -1.6
saved via ecostyles, attempt 1: g7_fig09_return_to_norm


alt.LayerChart(...)

In [50]:
def spread2(vals, gap_=0.9):                       # keeps end-of-line labels from colliding
    v = np.array(vals, float); out = v.copy()
    for i in range(1, len(out)): out[i] = max(out[i], out[i - 1] + gap_)
    return out - (out.mean() - v.mean())

# FIG 10: UK minus the median of the other six G7 countries, revenue and primary spending
yrs = p[p.year.between(2005, 2031)]
med = yrs[yrs.country != "United Kingdom"].groupby("year")[["GGR_NGDP", "prim_spend"]].median()   # median of the other six
uk_ = yrs[yrs.country == "United Kingdom"].set_index("year")[["GGR_NGDP", "prim_spend"]]
gap10 = (uk_ - med).rename(columns={"GGR_NGDP": "Revenue", "prim_spend": "Primary spending"})
d10 = gap10.reset_index().melt("year", var_name="kind", value_name="v")
xs10 = alt.Scale(domain=[2005, 2038], zero=False)
ax10 = alt.Axis(format="d", values=[2005, 2010, 2015, 2020, 2025, 2030], grid=False)
proj10 = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
    x=alt.X("a:Q", scale=xs10, title=None, axis=ax10), x2="b:Q")
zero10 = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[1, 5], color="#94a3b8").encode(y="y:Q")
ln10 = alt.Chart(d10).mark_line(strokeWidth=1.8).encode(
    x=alt.X("year:Q", title=None, scale=xs10, axis=ax10),
    y=alt.Y("v:Q", title="pp of GDP vs rest-of-G7 median", scale=alt.Scale(domain=[-9, 1])),
    color=alt.Color("kind:N", legend=None, sort=["Revenue", "Primary spending"]))
last10 = d10[d10.year == 2031].sort_values("v").copy()
last10["ylab"] = spread2(last10.v.values, gap_=0.8)
lab10 = alt.Chart(last10).mark_text(align="left", dx=4, fontSize=11).encode(
    x=alt.X("year:Q", scale=xs10), y="ylab:Q", text="kind:N",
    color=alt.Color("kind:N", legend=None, sort=["Revenue", "Primary spending"]))
fig10 = (proj10 + zero10 + ln10 + lab10).properties(width=W, height=H, title="UK relative to the rest of the G7, % of GDP")
save(fig10, "g7_fig10_uk_vs_rest_of_g7"); dump(gap10, "g7_fig10_uk_vs_rest_of_g7")
print(gap10.loc[[2007, 2019, 2025, 2031]].round(1))

# FIG 11: net interest bill by country
d11 = p[p.year.between(2005, 2031)][["country", "year", "interest"]].copy()
xs11 = alt.Scale(domain=[2005, 2038], zero=False)
ax11 = alt.Axis(format="d", values=[2005, 2010, 2015, 2020, 2025, 2030], grid=False)
proj11 = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
    x=alt.X("a:Q", scale=xs11, title=None, axis=ax11), x2="b:Q")
ln11 = alt.Chart(d11).mark_line(strokeWidth=1.8).encode(
    x=alt.X("year:Q", title=None, scale=xs11, axis=ax11), y=alt.Y("interest:Q", title="% of GDP"),
    color=alt.Color("country:N", legend=None))
last11 = d11[d11.year == 2031].sort_values("interest").copy()
last11["ylab"] = spread2(last11.interest.values, gap_=0.28)
lb11 = alt.Chart(last11).encode(x=alt.X("year:Q", scale=xs11), y="ylab:Q", text="country:N", color=alt.Color("country:N", legend=None))
lab11 = lb11.mark_text(align="left", dx=4, fontSize=11).transform_filter("datum.country != 'United Kingdom'")
lab11uk = lb11.mark_text(align="left", dx=4, fontSize=11, fontWeight="bold").transform_filter("datum.country == 'United Kingdom'")
fig11 = (proj11 + ln11 + lab11 + lab11uk).properties(width=W, height=H, title="Net interest bill, G7")
save(fig11, "g7_fig11_net_interest"); dump(d11.pivot(index="year", columns="country", values="interest"), "g7_fig11_net_interest")

# FIG 12: distance from stabilising debt, three-year averages (2023-25 vs 2029-31 IMF projection)
gp = p.pivot(index="year", columns="country", values="gap")
g12 = pd.DataFrame({"2023–25": gp.loc[2023:2025].mean(), "2029–31 (IMF projection)": gp.loc[2029:2031].mean()})
o12 = g12["2029–31 (IMF projection)"].sort_values(ascending=False).index.tolist()
l12 = g12.reset_index().melt("country", var_name="period", value_name="gap_")
rules12 = g12.reset_index().rename(columns={"2023–25": "a", "2029–31 (IMF projection)": "b"})
xs12 = alt.Scale(domain=[-4, 8.5])
rl = alt.Chart(rules12).mark_rule(color="#94a3b8", strokeWidth=1.5).encode(
    y=alt.Y("country:N", sort=o12, title=None), x=alt.X("a:Q", title="Fiscal gap, pp of GDP", scale=xs12), x2="b:Q")
dt = alt.Chart(l12).mark_point(filled=True, size=70, opacity=1).encode(
    y=alt.Y("country:N", sort=o12), x=alt.X("gap_:Q", scale=xs12),
    color=alt.Color("period:N", title=None, sort=["2023–25", "2029–31 (IMF projection)"],
                    legend=alt.Legend(orient="top", direction="horizontal")))
z12 = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(strokeDash=[1, 5], color="#94a3b8").encode(x="x:Q")
fig12 = (z12 + rl + dt).properties(width=W, height=H, title="Distance from stabilising debt, now vs IMF baseline")
save(fig12, "g7_fig12_fiscal_gap_now_vs_2031"); dump(g12, "g7_fig12_fiscal_gap_now_vs_2031")
print(g12.round(2))
print(d11[d11.year.isin([2007, 2019, 2025, 2031])].pivot(index="year", columns="country", values="interest").round(1))
fig12

saved via ecostyles, attempt 1: g7_fig10_uk_vs_rest_of_g7
code  Revenue  Primary spending
year                           
2007     -6.3              -2.9
2019     -7.8              -5.8
2025     -6.8              -4.8
2031     -2.5              -3.5
saved via ecostyles, attempt 1: g7_fig11_net_interest
saved via ecostyles, attempt 1: g7_fig12_fiscal_gap_now_vs_2031
                2023–25  2029–31 (IMF projection)
country                                          
Canada             2.97                      2.06
France            -1.22                      0.13
Germany           -0.26                     -1.76
Italy              0.90                      1.07
Japan              7.42                      1.64
United Kingdom    -0.61                      1.56
United States     -1.04                     -2.68
country  Canada  France  Germany  Italy  Japan  United Kingdom  United States
year                                                                         
2007        0.5     2.5   

alt.LayerChart(...)

In [51]:
raw = pd.read_csv(RAW, encoding="utf-8-sig")
raw["code"] = raw.SERIES_CODE.str.split(".").str[1]
yrs_ = [c for c in raw.columns if c.isdigit()]
maxdiff, n = 0.0, 0
for code in CODES:
    r = raw[raw.code == code].set_index("COUNTRY")[yrs_].apply(pd.to_numeric, errors="coerce")
    for c in r.index:
        for y in range(2000, 2032):
            a = r.loc[c, str(y)]; b = p[(p.country == c) & (p.year == y)][code]
            b = b.iloc[0] if len(b) else np.nan
            if pd.notna(a) and pd.notna(b): maxdiff = max(maxdiff, abs(a - b)); n += 1
print(f"{n} values compared against the raw CSV; max abs difference = {maxdiff}")

2010 values compared against the raw CSV; max abs difference = 0.0


In [52]:
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)   # show every column
# Inputs typed from IMF Fiscal Monitor, April 2026, Table A23 (pp of GDP; r-g in percentage points)
A23 = pd.DataFrame({
    "pension_5y":  {"Canada": 0.4, "France": 0.1, "Germany": 0.3, "Italy": 0.6, "Japan": 0.2, "United Kingdom": 0.0, "United States": 0.5},
    "health_5y":   {"Canada": 0.6, "France": 0.6, "Germany": 0.4, "Italy": 0.3, "Japan": 0.8, "United Kingdom": 0.7, "United States": 2.3},
    "pension_npv": {"Canada": 9.4, "France": -2.1, "Germany": 10.0, "Italy": 13.3, "Japan": 31.3, "United Kingdom": 6.8, "United States": 13.4},
    "health_npv":  {"Canada": 34.6, "France": 31.0, "Germany": 30.4, "Italy": 21.5, "Japan": 35.0, "United Kingdom": 41.0, "United States": 100.9},
    "rg":          {"Canada": -0.5, "France": -0.2, "Germany": -1.0, "Italy": 0.3, "Japan": -1.5, "United Kingdom": 0.2, "United States": -0.2},
})
A23["age_5y_imf"] = A23.pension_5y + A23.health_5y          # IMF change in age-related spending, 2025-30
A23["age_npv"] = A23.pension_npv + A23.health_npv           # IMF NPV of the change, 2025-50
START, END = 2026, 2050

# annual path of age-related spending: a straight-line rise whose net present value over 2025-50
# (discounted at 1% above growth, as in the IMF table) equals the IMF's pension + health NPV
v = 1 / 1.01
t = np.arange(1, 26)                                         # 2026 ... 2050
sum_tv = (t * v**t).sum()
A23["age_slope"] = A23.age_npv / sum_tv                      # pp of GDP added per year
A23["age_rise_2050"] = A23.age_slope * 25

# checks: (1) does the path reproduce the IMF NPV?  (2) how close is its first five years to the IMF's own 2025-30 change?
A23["npv_check"] = [(A23.age_slope[c] * t * v**t).sum() for c in A23.index]
A23["five_year_implied"] = A23.age_slope * 5
print(A23[["age_npv", "npv_check", "age_5y_imf", "five_year_implied", "age_slope", "age_rise_2050", "rg"]].round(2))
dump(A23, "g7_projection_inputs")

                age_npv  npv_check  age_5y_imf  five_year_implied  age_slope  age_rise_2050   rg
Canada             44.0       44.0         1.0               0.80       0.16           4.00 -0.5
France             28.9       28.9         0.7               0.53       0.11           2.63 -0.2
Germany            40.4       40.4         0.7               0.73       0.15           3.67 -1.0
Italy              34.8       34.8         0.9               0.63       0.13           3.16  0.3
Japan              66.3       66.3         1.0               1.21       0.24           6.03 -1.5
United Kingdom     47.8       47.8         0.7               0.87       0.17           4.35  0.2
United States     114.3      114.3         2.8               2.08       0.42          10.39 -0.2


In [53]:

ADJ_YEARS, ADJ_PER_YEAR = 5, 1.0
SCEN = ["Unchanged policy", "Gradual adjustment", "Adjustment + ageing frozen"]

def project(country, rg_shift=0.0, adj_per_year=ADJ_PER_YEAR):
    s = p[p.country == country].set_index("year")
    d, pb0 = s.loc[START, "GGXWDG_NGDP"], s.loc[START, "pb"]          # IMF 2026 gross debt and primary balance
    delta = (A23.loc[country, "rg"] + rg_shift) / 100                 # (r - g), held constant
    slope = A23.loc[country, "age_slope"]
    paths = {k: [d] for k in SCEN}; dd = {k: d for k in SCEN}
    for yr in range(START + 1, END + 1):
        age = slope * (yr - START)
        adj = min(yr - START, ADJ_YEARS) * adj_per_year
        pbs = {SCEN[0]: pb0 - age, SCEN[1]: pb0 + adj - age, SCEN[2]: pb0 + adj}
        for k in SCEN:
            dd[k] = dd[k] * (1 + delta) - pbs[k]
            paths[k].append(dd[k])
    return pd.DataFrame(paths, index=range(START, END + 1))

proj = {c: project(c) for c in A23.index}
proj_hi = {c: project(c, rg_shift=1.0) for c in A23.index}           # r-g one point higher
proj_lo = {c: project(c, rg_shift=-1.0) for c in A23.index}          # r-g one point lower

summ = pd.DataFrame({
    "debt_2026": {c: proj[c].iloc[0, 0] for c in proj},
    "unchanged_2035": {c: proj[c].loc[2035, SCEN[0]] for c in proj},
    "unchanged_2050": {c: proj[c].loc[2050, SCEN[0]] for c in proj},
    "adjustment_2050": {c: proj[c].loc[2050, SCEN[1]] for c in proj},
    "adj_ageing_frozen_2050": {c: proj[c].loc[2050, SCEN[2]] for c in proj},
    "unchanged_2050_rg_minus1": {c: proj_lo[c].loc[2050, SCEN[0]] for c in proj},
    "unchanged_2050_rg_plus1": {c: proj_hi[c].loc[2050, SCEN[0]] for c in proj},
}).round(1)
print(summ)

# sanity check against the IMF baseline: where the IMF has debt in 2031 vs our unchanged-policy path
chk = pd.DataFrame({
    "IMF_debt_2031": {c: p[(p.country == c) & (p.year == 2031)].GGXWDG_NGDP.iloc[0] for c in proj},
    "unchanged_2031": {c: proj[c].loc[2031, SCEN[0]] for c in proj},
    "adjustment_2031": {c: proj[c].loc[2031, SCEN[1]] for c in proj},
    "IMF_pb_2031": {c: p[(p.country == c) & (p.year == 2031)].pb.iloc[0] for c in proj},
    "pb_2026_held": {c: p[(p.country == c) & (p.year == 2026)].pb.iloc[0] for c in proj},
}).round(1)
print(chk)
dump(summ, "g7_projection_summary"); dump(chk, "g7_projection_vs_imf_2031"); dump(pd.concat(proj, axis=1), "g7_projection_paths")

                debt_2026  unchanged_2035  unchanged_2050  adjustment_2050  adj_ageing_frozen_2050  unchanged_2050_rg_minus1  unchanged_2050_rg_plus1
Canada              110.7           134.4           199.5             95.1                    48.9                     169.3                    236.5
France              118.4           145.2           207.5             99.8                    68.7                     174.3                    248.3
Germany              64.6            90.0           152.4             53.3                    12.4                     132.3                    176.7
Italy               138.4           140.7           168.0             54.5                    15.6                     135.5                    208.7
Japan               204.4           203.5           242.1            147.9                    83.2                     203.2                    290.2
United Kingdom      103.6           124.5           191.9             79.5                    26.5  

In [54]:
def required_pb(d0, target, rg, n):
    a = 1 + rg / 100
    return (d0 * a**n - target) / n if abs(a - 1) < 1e-9 else (d0 * a**n - target) * (a - 1) / (a**n - 1)

req = []
for c in A23.index:
    s = p[p.country == c].set_index("year"); d0 = s.loc[START, "GGXWDG_NGDP"]
    row = {"country": c, "debt_2026": d0, "debt_2019": s.loc[2019, "GGXWDG_NGDP"], "debt_2007": s.loc[2007, "GGXWDG_NGDP"]}
    for tgt_year in (2019, 2007):
        for n in (5, 10, 20):
            row[f"to_{tgt_year}_over_{n}y"] = required_pb(d0, s.loc[tgt_year, "GGXWDG_NGDP"], A23.loc[c, "rg"], n)
    row["IMF_avg_pb_2027_31"] = s.loc[2027:2031, "pb"].mean()
    req.append(row)
req = pd.DataFrame(req).set_index("country").round(1)
print(req)
dump(req, "g7_required_primary_balance")

                debt_2026  debt_2019  debt_2007  to_2019_over_5y  to_2019_over_10y  to_2019_over_20y  to_2007_over_5y  to_2007_over_10y  to_2007_over_20y  IMF_avg_pb_2027_31
country                                                                                                                                                                      
Canada              110.7       90.1       67.2              3.6               1.6               0.5              8.2               3.9               1.7                -1.3
France              118.4       98.2       65.5              3.8               1.8               0.8             10.4               5.1               2.5                -1.0
Germany              64.6       58.7       63.7              0.6              -0.0              -0.3             -0.5              -0.5              -0.6                -2.6
Italy               138.4      133.9      103.5              1.3               0.9               0.6              7.4             

In [55]:

paper_pb2011 = {"France": -5.1, "Germany": -2.0, "Italy": 0.0, "Japan": -8.0, "United Kingdom": -9.0, "United States": -7.1}
rows = []
for c, f in paper_pb2011.items():
    s = p[p.country == c].set_index("year")
    rows.append({"country": c, "pb_2011_paper_forecast": f, "pb_2011_actual": s.loc[2011, "pb"], "pb_2019_actual": s.loc[2019, "pb"],
                 "debt_2019": s.loc[2019, "GGXWDG_NGDP"], "debt_2020": s.loc[2020, "GGXWDG_NGDP"], "debt_2025": s.loc[2025, "GGXWDG_NGDP"]})
retro = pd.DataFrame(rows).set_index("country").round(1)
retro["paper_warning_debt"] = pd.Series({"France": ">150", "Germany": "n/s", "Italy": ">150", "Japan": ">300", "United Kingdom": ">200", "United States": ">150"})
print(retro)
dump(retro, "g7_retrospective_2010_paper")

                pb_2011_paper_forecast  pb_2011_actual  pb_2019_actual  debt_2019  debt_2020  debt_2025 paper_warning_debt
country                                                                                                                   
France                            -5.1            -2.8            -0.9       98.2      114.9      116.0               >150
Germany                           -2.0             1.2             1.9       58.7       68.0       62.9                n/s
Italy                              0.0             0.9             1.7      133.9      154.4      137.1               >150
Japan                             -8.0            -7.8            -2.4      206.3      228.8      206.5               >300
United Kingdom                    -9.0            -4.6            -1.0       84.9      104.8      102.3               >200
United States                     -7.1            -7.4            -3.5      108.8      132.6      123.9               >150


In [56]:
def spread2(vals, gap_=0.9):                       
    v = np.array(vals, float); out = v.copy()
    for i in range(1, len(out)): out[i] = max(out[i], out[i - 1] + gap_)
    return out - (out.mean() - v.mean())

SC = ["Outturn"] + SCEN
def scenario_chart(country):
    s = p[p.country == country].set_index("year")
    hist = s.loc[2000:2026, "GGXWDG_NGDP"].rename("v").reset_index().assign(series="Outturn")
    sc_ = proj[country].reset_index().melt("index", var_name="series", value_name="v").rename(columns={"index": "year"})
    d = pd.concat([hist, sc_[["year", "series", "v"]]], ignore_index=True)
    xs = alt.Scale(domain=[2000, 2070], zero=False)
    ax = alt.Axis(format="d", values=[2000, 2010, 2020, 2030, 2040, 2050], grid=False)
    ln = alt.Chart(d).mark_line(strokeWidth=1.8).encode(
        x=alt.X("year:Q", title=None, scale=xs, axis=ax), y=alt.Y("v:Q", title="% of GDP"),
        color=alt.Color("series:N", legend=None, sort=SC))
    end = d[d.year == 2050].sort_values("v").copy()
    rng = max(d.v.max() - d.v.min(), 1)
    end["ylab"] = spread2(end.v.values, gap_=rng * 0.07)
    lab = alt.Chart(end).mark_text(align="left", dx=4, fontSize=11).encode(
        x=alt.X("year:Q", scale=xs), y="ylab:Q", text="series:N", color=alt.Color("series:N", legend=None, sort=SC))
    return (ln + lab).properties(width=W, height=H, title=f"{country}: gross debt under three scenarios")

for c in A23.index:
    fig = scenario_chart(c)
    save(fig, f"g7_fig13_projection_{c.lower().replace(' ', '_')}")
fig

saved via ecostyles, attempt 1: g7_fig13_projection_canada
saved via ecostyles, attempt 1: g7_fig13_projection_france
saved via ecostyles, attempt 1: g7_fig13_projection_germany
saved via ecostyles, attempt 1: g7_fig13_projection_italy
saved via ecostyles, attempt 1: g7_fig13_projection_japan
saved via ecostyles, attempt 1: g7_fig13_projection_united_kingdom
saved via ecostyles, attempt 1: g7_fig13_projection_united_states


alt.LayerChart(...)

In [57]:
expected = {"Canada": 199.5, "France": 207.5, "Germany": 152.4, "Italy": 168.0, "Japan": 242.1, "United Kingdom": 191.9, "United States": 328.8}
chk = pd.DataFrame({"yours": summ.unchanged_2050, "expected": pd.Series(expected)})
chk["diff"] = (chk.yours - chk.expected).round(1)
print(chk)
print(sorted(f for f in os.listdir(OUT) if "fig13" in f))

                yours  expected  diff
Canada          199.5     199.5   0.0
France          207.5     207.5   0.0
Germany         152.4     152.4   0.0
Italy           168.0     168.0   0.0
Japan           242.1     242.1   0.0
United Kingdom  191.9     191.9   0.0
United States   328.8     328.8   0.0
['g7_fig13_projection_canada.json', 'g7_fig13_projection_canada.png', 'g7_fig13_projection_canada.svg', 'g7_fig13_projection_france.json', 'g7_fig13_projection_france.png', 'g7_fig13_projection_france.svg', 'g7_fig13_projection_germany.json', 'g7_fig13_projection_germany.png', 'g7_fig13_projection_germany.svg', 'g7_fig13_projection_italy.json', 'g7_fig13_projection_italy.png', 'g7_fig13_projection_italy.svg', 'g7_fig13_projection_japan.json', 'g7_fig13_projection_japan.png', 'g7_fig13_projection_japan.svg', 'g7_fig13_projection_united_kingdom.json', 'g7_fig13_projection_united_kingdom.png', 'g7_fig13_projection_united_kingdom.svg', 'g7_fig13_projection_united_states.json', 'g7_fig13_proj

In [58]:
SC = ["Outturn"] + SCEN
PANEL_W, PANEL_H = 210, 150
COLOUR = alt.Color("series:N", scale=alt.Scale(domain=SC), legend=None)

def panel(country, first_in_row, shared_y, ymax):
    s = p[p.country == country].set_index("year")
    hist = s.loc[2000:2026, "GGXWDG_NGDP"].rename("v").reset_index().assign(series="Outturn")
    sc_ = proj[country].reset_index().melt("index", var_name="series", value_name="v").rename(columns={"index": "year"})
    d = pd.concat([hist, sc_[["year", "series", "v"]]], ignore_index=True)
    top = ymax if shared_y else float(np.ceil(d.v.max() * 1.06 / 10) * 10)
    xs, ys = alt.Scale(domain=[2000, 2050], zero=False), alt.Scale(domain=[0, top])
    ax = alt.Axis(format="d", values=[2000, 2025, 2050], grid=False)
    yax = alt.Axis() if (first_in_row or not shared_y) else alt.Axis(labels=False, ticks=False, domain=False)
    band = alt.Chart(pd.DataFrame({"a": [2026], "b": [2050]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
        x=alt.X("a:Q", scale=xs, title=None, axis=ax), x2="b:Q")
    ln = alt.Chart(d).mark_line(strokeWidth=1.6).encode(
        x=alt.X("year:Q", scale=xs, title=None, axis=ax), y=alt.Y("v:Q", scale=ys, title=None, axis=yax), color=COLOUR)
    return (band + ln).properties(width=PANEL_W, height=PANEL_H, title=alt.TitleParams(country, anchor="start", fontSize=12))

def key_cell():                                        # fills the free slot in the last row
    d = pd.DataFrame({"series": SC, "y": list(range(len(SC) - 1, -1, -1))})
    return alt.Chart(d).mark_text(align="left", fontSize=11).encode(
        x=alt.value(8), y=alt.Y("y:Q", axis=None, scale=alt.Scale(domain=[-0.6, len(SC) - 0.4])), text="series:N",
        color=COLOUR).properties(width=PANEL_W, height=PANEL_H)

def key_strip(width):                                  # used when the last row is full
    d = pd.DataFrame({"series": SC, "x": range(len(SC))})
    return alt.Chart(d).mark_text(align="left", fontSize=11).encode(
        x=alt.X("x:Q", axis=None, scale=alt.Scale(domain=[-0.02, len(SC) - 0.02])), y=alt.value(8), text="series:N",
        color=COLOUR).properties(width=width, height=16)

def scenario_grid(countries, ncol=4, shared_y=True):
    ymax = float(np.ceil(max(max(proj[c].max().max(), p[p.country == c].GGXWDG_NGDP.max()) for c in countries) * 1.05 / 50) * 50)
    cells = [panel(c, i % ncol == 0, shared_y, ymax) for i, c in enumerate(countries)]
    rows = [cells[i:i + ncol] for i in range(0, len(cells), ncol)]
    if len(rows[-1]) < ncol: rows[-1] = rows[-1] + [key_cell()]
    else: rows.append([key_strip(ncol * PANEL_W)])
    return alt.vconcat(*[alt.hconcat(*r) for r in rows]).properties(title="Gross debt under three scenarios, G7, % of GDP")

ORDER_7 = ["United Kingdom", "United States", "Japan", "Germany", "France", "Italy", "Canada"]
fig14 = scenario_grid(ORDER_7, ncol=4)                 # 4 on top, 3 below, key in the spare slot
save(fig14, "g7_fig14_projections_grid")

fig14

saved via ecostyles, attempt 1: g7_fig14_projections_grid


alt.VConcatChart(...)

In [59]:
# ---------- PATCH: DEFINES THE PIECES THE SUMMARY TABLE NEEDS (run once, before Cell 21) ----------
def required_pb(d0, target, rg, n):
    a = 1 + rg / 100
    return (d0 * a**n - target) / n if abs(a - 1) < 1e-9 else (d0 * a**n - target) * (a - 1) / (a**n - 1)

# BIS Working Paper 300 (2010): 2011 primary balance forecasts (Table 3 memo, OECD data) and debt warnings from the text
paper_pb2011 = {"France": -5.1, "Germany": -2.0, "Italy": 0.0, "Japan": -8.0, "United Kingdom": -9.0, "United States": -7.1}
paper_warning = {"France": ">150", "Germany": "not stated", "Italy": ">150", "Japan": ">300", "United Kingdom": ">200", "United States": ">150"}

def drivers_of_rise(country):
    """Splits the unchanged-policy rise in debt, 2026-2050, into three exact pieces."""
    s = p[p.country == country].set_index("year")
    d, pb0 = s.loc[START, "GGXWDG_NGDP"], s.loc[START, "pb"]
    delta, slope = A23.loc[country, "rg"] / 100, A23.loc[country, "age_slope"]
    init = ageing = snow = 0.0; dprev = d
    for yr in range(START + 1, END + 1):
        age = slope * (yr - START); pb = pb0 - age
        init += -pb0; ageing += age; snow += delta * dprev
        dprev = dprev * (1 + delta) - pb
    assert abs((dprev - d) - (init + ageing + snow)) < 1e-9 and abs(dprev - proj[country].loc[END, SCEN[0]]) < 1e-9
    return {"today's primary deficit held (-pb 2026 x 24 yrs)": init, "rising age-related spending": ageing,
            "interest-growth (r-g) on the debt stock": snow, "total rise in debt ratio, 2026-50": dprev - d}

In [60]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
ORDER = ["United Kingdom", "United States", "Japan", "Germany", "France", "Italy", "Canada"]

def first_year_above(country, thr):
    u, d0 = proj[country][SCEN[0]], proj[country].iloc[0, 0]
    if d0 > thr: return "already above"
    return int(u[u > thr].index[0]) if (u > thr).any() else "not by 2050"

# TABLE A: projections, 2026-2050
rows = []
for c in ORDER:
    s = p[p.country == c].set_index("year"); a = A23.loc[c]; d0 = s.loc[START, "GGXWDG_NGDP"]
    init, ageing, snow, total = drivers_of_rise(c).values()
    rows.append({
        ("Starting point", "Debt 2026"): d0,
        ("Starting point", "Primary balance 2026"): s.loc[START, "pb"],
        ("Starting point", "IMF debt 2031"): s.loc[2031, "GGXWDG_NGDP"],
        ("Debt in 2050 (% GDP)", "Unchanged policy"): proj[c].loc[END, SCEN[0]],
        ("Debt in 2050 (% GDP)", "Gradual adjustment"): proj[c].loc[END, SCEN[1]],
        ("Debt in 2050 (% GDP)", "Adjustment + ageing frozen"): proj[c].loc[END, SCEN[2]],
        ("Debt in 2050 (% GDP)", "Unchanged, r-g +1pp"): proj_hi[c].loc[END, SCEN[0]],
        ("Unchanged-policy rise, 2026-50 (pp)", "Today's deficit held"): init,
        ("Unchanged-policy rise, 2026-50 (pp)", "Ageing"): ageing,
        ("Unchanged-policy rise, 2026-50 (pp)", "Interest-growth"): snow,
        ("Unchanged-policy rise, 2026-50 (pp)", "Total"): total,
        ("Unchanged policy: first year above", "150%"): first_year_above(c, 150),
        ("Primary balance to return to 2019 debt in 10y", "Required"): required_pb(d0, s.loc[2019, "GGXWDG_NGDP"], a.rg, 10),
        ("Primary balance to return to 2019 debt in 10y", "IMF avg 2027-31"): s.loc[2027:2031, "pb"].mean(),
    })
summary_a = pd.DataFrame(rows, index=ORDER)
summary_a.columns = pd.MultiIndex.from_tuples(summary_a.columns)
summary_a = summary_a.round(1)

# TABLE B: what the 2010 BIS paper assumed and warned vs what happened
rows_b = {}
for c in [x for x in ORDER if x in paper_pb2011]:
    s = p[p.country == c].set_index("year")
    rows_b[c] = {"2011 primary balance: paper's forecast": paper_pb2011[c], "2011 primary balance: actual": s.loc[2011, "pb"],
                 "2019 primary balance: actual": s.loc[2019, "pb"], "Paper's debt warning (% GDP)": paper_warning[c],
                 "Actual debt 2020": s.loc[2020, "GGXWDG_NGDP"], "Actual debt 2025": s.loc[2025, "GGXWDG_NGDP"]}
summary_b = pd.DataFrame(rows_b).T
for col in summary_b.columns:
    if col != "Paper's debt warning (% GDP)": summary_b[col] = summary_b[col].astype(float).round(1)
dump(summary_a, "g7_summary_table_a_projections"); dump(summary_b, "g7_summary_table_b_retrospective")

show = display if "display" in globals() else print          # tables render nicely in a notebook
print("TABLE A: DEBT PROJECTIONS, % OF GDP (April 2026 IMF data; Fiscal Monitor ageing inputs)"); show(summary_a)
print("\nTABLE B: THE 2010 BIS PAPER AGAINST WHAT HAPPENED (six G7 economies in its sample)"); show(summary_b)

TABLE A: DEBT PROJECTIONS, % OF GDP (April 2026 IMF data; Fiscal Monitor ageing inputs)
               Starting point                                    Debt in 2050 (% GDP)                                                                   Unchanged-policy rise, 2026-50 (pp)                                \
                    Debt 2026 Primary balance 2026 IMF debt 2031     Unchanged policy Gradual adjustment Adjustment + ageing frozen Unchanged, r-g +1pp                Today's deficit held Ageing Interest-growth  Total   
United Kingdom          103.6                 -1.2         102.6                191.9               79.5                       26.5               229.2                                29.5   52.2             6.6   88.3   
United States           125.8                 -3.7         142.1                328.8              221.0                       98.2               381.6                                88.0  124.7            -9.8  203.0   
Japan                   204.

In [63]:
# ---------- CELL 22: FIG 15 - THE 2010 BIS WARNINGS AGAINST WHAT HAPPENED (needs Cells 0-1) ----------
# Thresholds are read from the TEXT of BIS WP300 (section 3): baseline debt "exceeding" these levels. Germany: no figure in the text.
paper_threshold = {"Japan": 300, "United Kingdom": 200, "France": 150, "Italy": 150, "United States": 150}
ORDER15 = ["Japan", "United Kingdom", "France", "Italy", "United States"]
LABELS = ["Actual debt 2020", "Paper's 2010 warning", "Actual debt 2025"]      # theme colours follow this order
XMAX = 350

rows = []
for c in ORDER15:
    s = p[p.country == c].set_index("year")
    rows.append({"country": c, "threshold": paper_threshold[c], "d2020": s.loc[2020, "GGXWDG_NGDP"], "d2025": s.loc[2025, "GGXWDG_NGDP"]})
d15 = pd.DataFrame(rows)
dump(d15.set_index("country"), "g7_fig15_2010_warnings_vs_outturn")
print(d15.round(1).to_string(index=False))

COL = alt.Color("what:N", scale=alt.Scale(domain=LABELS), legend=alt.Legend(title=None, orient="top", direction="horizontal", symbolType="circle", symbolOpacity=1))
ysort = alt.Y("country:N", sort=ORDER15, title=None)
xs = alt.Scale(domain=[0, XMAX])
warn = d15.assign(what="Paper's 2010 warning", top=XMAX)
warn["txt"] = ">" + warn.threshold.astype(int).astype(str)                # short label sits inside the shaded zone, clear of the tick and dots
zone = alt.Chart(warn).mark_bar(opacity=0.22, size=18).encode(
    y=ysort, x=alt.X("threshold:Q", scale=xs, title="Gross debt, % of GDP"), x2="top:Q", color=COL)
tick = alt.Chart(warn).mark_tick(thickness=3, size=24).encode(y=ysort, x=alt.X("threshold:Q", scale=xs), color=COL)
tlab = alt.Chart(warn).mark_text(align="right", dx=-6, fontSize=11).encode(
    y=ysort, x=alt.X("top:Q", scale=xs), text="txt:N", color=COL)
actual = pd.concat([d15.assign(what="Actual debt 2020", v=d15.d2020), d15.assign(what="Actual debt 2025", v=d15.d2025)])
try:                                   # nudge the two years apart vertically so close values stay visible (needs Altair 5+)
    pts = alt.Chart(actual).mark_point(filled=True, size=110, opacity=1).encode(
        y=ysort, x=alt.X("v:Q", scale=xs), color=COL,
        yOffset=alt.YOffset("what:N", scale=alt.Scale(domain=["Actual debt 2020", "Actual debt 2025"], paddingInner=0, paddingOuter=1)))
except TypeError:                      # older Altair: same dots, with a thin white edge instead
    pts = alt.Chart(actual).mark_point(filled=True, size=110, opacity=1, stroke="white", strokeWidth=1.5).encode(
        y=ysort, x=alt.X("v:Q", scale=xs), color=COL)
fig15 = (zone + tick + tlab + pts).properties(width=W, height=H, title="The 2010 BIS warning against what happened")
save(fig15, "g7_fig15_2010_warnings_vs_outturn")
fig15

       country  threshold  d2020  d2025
         Japan        300  228.8  206.5
United Kingdom        200  104.8  102.3
        France        150  114.9  116.0
         Italy        150  154.4  137.1
 United States        150  132.6  123.9
saved via ecostyles, attempt 1: g7_fig15_2010_warnings_vs_outturn


alt.LayerChart(...)

In [66]:
# ---------- CELL 23: FIG 16 - WHAT DRIVES THE UNCHANGED-POLICY RISE IN DEBT, 2026-2050 (needs Cell 15 or the patch) ----------
NAMES16 = ["Rising age-related spending", "Today's deficit held", "Interest-growth effect", "Net rise in debt ratio"]   # theme colours follow this order
rows = []
for c in A23.index:
    init, ageing, snow, total = drivers_of_rise(c).values()
    rows.append({"country": c, NAMES16[0]: ageing, NAMES16[1]: init, NAMES16[2]: snow, NAMES16[3]: total})
t16 = pd.DataFrame(rows).set_index("country")
order16 = ["United Kingdom"] + t16.drop("United Kingdom").sort_values(NAMES16[3], ascending=False).index.tolist()
dump(t16.round(1), "g7_fig16_unchanged_policy_drivers")
print(t16.loc[order16].round(1).to_string())

XMIN, XMAX = -100, 250
long16 = t16.reset_index().melt("country", value_vars=NAMES16[:3], var_name="part", value_name="v")
long16["ord"] = long16.part.map({n: i for i, n in enumerate(NAMES16[:3])})
tot16 = t16.reset_index()[["country", NAMES16[3]]].rename(columns={NAMES16[3]: "v"}).assign(part=NAMES16[3])

COL16 = alt.Color("part:N", scale=alt.Scale(domain=NAMES16), legend=alt.Legend(title=None, orient="top", direction="horizontal", columns=2, symbolType="circle", symbolOpacity=1))
y16 = alt.Y("country:N", sort=order16, title=None)
xs16 = alt.Scale(domain=[XMIN, XMAX])
ax16 = alt.Axis(values=list(range(-100, 251, 50)), title="Change in gross debt, 2026-2050, percentage points of GDP")
bars = alt.Chart(long16).mark_bar(size=16).encode(
    y=y16, x=alt.X("v:Q", scale=xs16, axis=ax16), color=COL16, order=alt.Order("ord:Q"))
zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color="#475569", strokeWidth=1).encode(x=alt.X("x:Q", scale=xs16))
dia = alt.Chart(tot16).mark_point(shape="diamond", filled=True, size=110, opacity=1, stroke="white", strokeWidth=1).encode(
    y=y16, x=alt.X("v:Q", scale=xs16), color=COL16)
fig16 = (bars + zero + dia).properties(width=W, height=H, title="What drives the rise in debt under unchanged policy")
save(fig16, "g7_fig16_unchanged_policy_drivers")
fig16

                Rising age-related spending  Today's deficit held  Interest-growth effect  Net rise in debt ratio
country                                                                                                          
United Kingdom                         52.2                  29.5                     6.6                    88.3
United States                         124.7                  88.0                    -9.8                   203.0
France                                 31.5                  65.1                    -7.5                    89.1
Canada                                 48.0                  58.3                   -17.6                    88.8
Germany                                44.1                  68.0                   -24.3                    87.8
Japan                                  72.3                  41.7                   -76.3                    37.8
Italy                                  38.0                 -18.9                    10.

alt.LayerChart(...)

In [65]:
# ---------- CELL 24: FIG 17 - PRIMARY BALANCE NEEDED TO RETURN TO A TARGET DEBT LEVEL VS THE IMF BASELINE (needs Cell 15 or the patch) ----------
TARGET_YEAR = 2019                      # the paper used 2007; change here to switch
HORIZONS = (5, 10, 20)
NAMES17 = [f"Needed over {n} years" for n in HORIZONS] + ["IMF baseline, 2027-31 average"]   # theme colours follow this order
rows = []
for c in A23.index:
    s = p[p.country == c].set_index("year"); d0, tv = s.loc[START, "GGXWDG_NGDP"], s.loc[TARGET_YEAR, "GGXWDG_NGDP"]
    r = {NAMES17[i]: required_pb(d0, tv, A23.loc[c, "rg"], n) for i, n in enumerate(HORIZONS)}
    r[NAMES17[3]] = s.loc[2027:2031, "pb"].mean(); r["country"] = c; rows.append(r)
t17 = pd.DataFrame(rows).set_index("country")
t17["shortfall_vs_10y"] = t17[NAMES17[3]] - t17[NAMES17[1]]          # IMF baseline minus the 10-year requirement (negative = falls short)
order17 = ["United Kingdom"] + t17.drop("United Kingdom").sort_values("shortfall_vs_10y").index.tolist()
dump(t17.round(1), "g7_fig17_required_primary_balance")
print(t17.loc[order17].round(1).to_string())

long17 = t17.reset_index().melt("country", value_vars=NAMES17, var_name="what", value_name="v")
span17 = t17.reset_index().assign(lo=t17[NAMES17[:3]].min(axis=1).values, hi=t17[NAMES17[:3]].max(axis=1).values)
COL17 = alt.Color("what:N", scale=alt.Scale(domain=NAMES17), legend=alt.Legend(title=None, orient="top", direction="horizontal", columns=2, symbolType="circle", symbolOpacity=1))
y17 = alt.Y("country:N", sort=order17, title=None)
xs17 = alt.Scale(domain=[-5, 5])
span = alt.Chart(span17).mark_rule(strokeWidth=4, opacity=0.25, color="#94a3b8").encode(
    y=y17, x=alt.X("lo:Q", scale=xs17, title="Constant primary balance, % of GDP"), x2="hi:Q")
zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color="#475569", strokeWidth=1).encode(x=alt.X("x:Q", scale=xs17))
need = alt.Chart(long17[long17.what != NAMES17[3]]).mark_point(filled=True, size=90, opacity=1).encode(
    y=y17, x=alt.X("v:Q", scale=xs17), color=COL17)
imf = alt.Chart(long17[long17.what == NAMES17[3]]).mark_point(shape="diamond", filled=True, size=170, opacity=1, stroke="white", strokeWidth=1).encode(
    y=y17, x=alt.X("v:Q", scale=xs17), color=COL17)
fig17 = (span + zero + need + imf).properties(width=W, height=H, title=f"Primary balance needed to return to the {TARGET_YEAR} debt level")
save(fig17, "g7_fig17_required_primary_balance")
fig17

                Needed over 5 years  Needed over 10 years  Needed over 20 years  IMF baseline, 2027-31 average  shortfall_vs_10y
country                                                                                                                         
United Kingdom                  3.9                   2.1                   1.1                            0.7              -1.3
United States                   3.2                   1.5                   0.6                           -3.4              -4.8
Canada                          3.6                   1.6                   0.5                           -1.3              -2.9
France                          3.8                   1.8                   0.8                           -1.0              -2.8
Germany                         0.6                  -0.0                  -0.3                           -2.6              -2.6
Italy                           1.3                   0.9                   0.6                  

alt.LayerChart(...)